In [48]:
import os
import pandas as pd
from torchvision.io import read_image
from torch.utils.data import Dataset
import os
import torch
from torch.utils.data import Dataset
import torchvision
import torchvision.transforms as transforms
import pandas as pd
from vit import ViT as vit 
from PIL import Image
import math as math
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models

class CustomImageDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = read_image(img_path)
        label = self.img_labels.iloc[idx, 1]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label

In [32]:
templist = []
for file in os.listdir("/home/bqtx/Documents/VLSI/ir_drop_ml/training_data/png-files/input"):
    templist.append(file)

In [33]:
df = pd.DataFrame(templist)
print(df.sort_values(by=0))

                      0
125      00_current.png
17      00_eff_dist.png
84   00_pdn_density.png
95       01_current.png
221     01_eff_dist.png
..                  ...
212     96_eff_dist.png
156  96_pdn_density.png
232      99_current.png
38      99_eff_dist.png
6    99_pdn_density.png

[252 rows x 1 columns]


In [73]:
# Custom Padding Transformer (FIX THIS)
class PadToSize:
    def __init__(self, target):
        self.target = target

    def __call__(self, img):
        # Calculate padding sizes

        colors, width, height = img.size()
        pad_left = math.floor((max(0, self.target[0] - width)/2))
        pad_right = math.ceil(((max(0, self.target[0]-width))/2))-1
        pad_top = math.floor((max(0, self.target[1] - height)/2))+1
        pad_bot = math.ceil((max(0,self.target[1] - height)/2))

        padding = (pad_left, pad_top, pad_right, pad_bot)
        if self.target[0]-width > width or self.target[1]-height > height:
            return transforms.functional.pad(img, padding, fill=0, padding_mode='constant')
        else:
            return transforms.functional.pad(img, padding, fill=0, padding_mode='reflect')
        
class ConvertToFloat32(object):
    def __call__(self, tensor):
        return tensor.to(torch.float32)

In [74]:
target_size = (930, 930)
pipeline= transforms.Compose([
    #transforms.ToTensor(),  # Convert image to tensor
    transforms.Lambda(lambda x: x[:3]),
    PadToSize(target_size),  # Pad the image to the target size
    ConvertToFloat32()
    #transforms.ToPILImage(), 
])

In [83]:
class StackedImagesDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.image_files = sorted(os.listdir(image_dir))  # Sorted to align image order
        self.label_files = sorted(os.listdir(label_dir))  # Assuming labels have the same order
        self.transform = transform
    
    def __len__(self):
        return len(self.label_files) # based off of the label amount
    
    def __getitem__(self, idx):
        # Get 3 consecutive images for stacking (you could choose any other strategy here)
        names = []

        img1_path = os.path.join(self.image_dir, self.image_files[3*idx])
        img2_path = os.path.join(self.image_dir, self.image_files[3*idx+1])
        img3_path = os.path.join(self.image_dir, self.image_files[3*idx+2])
        
        # Load images
        img1 = read_image(img1_path)
        img2 = read_image(img2_path)
        img3 = read_image(img3_path)
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)
            img3 = self.transform(img3)
        
        # Stack the 3 images along the channel dimension (depth-wise)
        stacked_images = torch.cat([img1,img2,img3])
        
        # Load corresponding label image
        label_path = os.path.join(self.label_dir, self.label_files[idx])  # Label corresponding to last image in stack
        label = read_image(label_path)
        if self.transform:
            label = self.transform(label)  # Apply transformation if needed
        
        names.append(self.image_files[3*idx])
        names.append(self.image_files[3*idx+1])
        names.append(self.image_files[3*idx+2])
        names.append(self.label_files[idx])

        return stacked_images, label #, idx, names

In [84]:
image_dataset = StackedImagesDataset(image_dir="/home/bqtx/Documents/VLSI/ir_drop_ml/training_data/png-files/input", label_dir= "/home/bqtx/Documents/VLSI/ir_drop_ml/training_data/png-files/ir_drop", transform=pipeline)

In [85]:
for image, label in image_dataset:
    print(image.size())
    print(label.size())
    #print(names)

torch.Size([9, 930, 930])
torch.Size([3, 930, 930])
torch.Size([9, 930, 930])
torch.Size([3, 930, 930])
torch.Size([9, 930, 930])
torch.Size([3, 930, 930])
torch.Size([9, 930, 930])
torch.Size([3, 930, 930])
torch.Size([9, 930, 930])
torch.Size([3, 930, 930])
torch.Size([9, 930, 930])
torch.Size([3, 930, 930])
torch.Size([9, 930, 930])
torch.Size([3, 930, 930])
torch.Size([9, 930, 930])
torch.Size([3, 930, 930])
torch.Size([9, 930, 930])
torch.Size([3, 930, 930])
torch.Size([9, 930, 930])
torch.Size([3, 930, 930])
torch.Size([9, 930, 930])
torch.Size([3, 930, 930])
torch.Size([9, 930, 930])
torch.Size([3, 930, 930])
torch.Size([9, 930, 930])
torch.Size([3, 930, 930])
torch.Size([9, 930, 930])
torch.Size([3, 930, 930])
torch.Size([9, 930, 930])
torch.Size([3, 930, 930])
torch.Size([9, 930, 930])
torch.Size([3, 930, 930])
torch.Size([9, 930, 930])
torch.Size([3, 930, 930])
torch.Size([9, 930, 930])
torch.Size([3, 930, 930])
torch.Size([9, 930, 930])
torch.Size([3, 930, 930])
torch.Size([

In [86]:
train_dataset, test_dataset = torch.utils.data.random_split(image_dataset, [0.8, 0.2])

In [96]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [101]:
model = vit(image_size = (930,930), patch_size = (15,15), num_classes = 2, dim = 3, depth = 1, heads = 4, mlp_dim = 10, channels=9) 

In [102]:
test = torch.rand(1,9,930,930)
for image, label in image_dataset:
   model(test)
   break
# only works if the images are batched together

In [104]:
criterion = nn.L1Loss()  # For multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Step 4: Training Loop
epochs = 1  # Number of epochs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in tqdm(range(epochs)):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()  # Zero the gradients before the backward pass
        print(inputs.size())
        outputs = model(inputs)  # Forward pass
        loss = criterion(outputs, labels)  # Calculate the loss
        loss.backward()  # Backward pass
        optimizer.step()  # Optimize the model

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # Print statistics for the current epoch
    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Loss: {running_loss/len(train_dataloader):.4f}")
    print(f"Accuracy: {100 * correct/total:.2f}%")

    # Step 5: Evaluate on Test Data
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0
    with torch.no_grad():  # No need to track gradients during evaluation
        for inputs, labels in test_dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"Test Accuracy: {100 * correct/total:.2f}%")
    print("-" * 50)

print("Training Finished!")

  0%|          | 0/1 [00:00<?, ?it/s]/home/bqtx/anaconda3/envs/ir_drop_ml/lib/python3.10/site-packages/torch/nn/modules/loss.py:101: UserWarning: Using a target size (torch.Size([1, 3, 930, 930])) that is different to the input size (torch.Size([1, 2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
  0%|          | 0/1 [00:00<?, ?it/s]

torch.Size([1, 9, 930, 930])


RuntimeError: The size of tensor a (2) must match the size of tensor b (930) at non-singleton dimension 3